In [1]:
# ignore warnings
import warnings
warnings.filterwarnings("ignore")

### Import necessary stuff from crewai and ollama

In [28]:
import ollama
from crewai import Agent,Task, Crew, LLM
from IPython.display import Markdown

### Setting up the LLM to be used, we are going to use a local LLM hosted by OLLAMA

In [8]:
LOCAL_LLM = LLM(
    model="ollama/llama3.1:8b",
    base_url="http://localhost:11434"
    )

# Setting up the Agents

Each Agent needs to have a Role, Goal, Backstory
Other config options like allow_delegation, verbose, llm can be set

## Planner Agent

In [20]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    allow_delegation=False,
    verbose=True,
    llm=LOCAL_LLM)

## Writer Agent

In [21]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provided by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provided by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    verbose=True,
    llm=LOCAL_LLM
)

## Editor Agent

In [22]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    verbose=True,
    llm=LOCAL_LLM
)

### Setting up tasks for the agents created 

For each task - description, expected_output, agent needs to be defined

In [23]:
# Task for the planner agent, plan task
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

In [24]:
# Task setup for the writer agent
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

In [25]:
# Task setup for the editor agent
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

### Setup the Crew

Tasks in the crew are going to be executed sequentially. One tasks output is consumed by the other as input  
verbose = True allows for detailed logging of each task's execution.

In [26]:
crew = Crew(
    agents=[planner,writer, editor],
    tasks=[plan,write,edit],
    verbose=True
)

## Running the crew

In [37]:
result = crew.kickoff(inputs={"topic" : "Artificial Intelligence"})

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 22f09677-4656-48b9-9aa9-9f94c651ccf9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🚀 Crew: crew
└── 📋 Task: 99e58ec7-290c-408f-81ac-e171db0c197d
       Status: Executing Task...

🚀 Crew: crew
└── 📋 Task: 99e58ec7-290c-408f-81ac-e171db0c197d
       Status: Executing Task...
    └── 🤖 Agent: Content Planner
            Status: In Progress

# Agent: Content Planner
## Task: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


# Agent: Content Planner
## Final Answer: 
**Content Plan Document for Artificial Intelligence**

**I. Introduction**

* Hook: "Artificial Intelligence (AI) is transforming industries worldwide, from healthcare to finance, and beyond."
* Brief overview of AI's impact on society
* Thesis statement: "In this article, we'll explore the latest trends, key players, and noteworthy news in AI, helping you understand its potential and challenges."

**II. Audience Analysis**

* Demographics:
	+ Age: 25-45 years old
	+ Education: Bachelor's degree or higher in a STEM field (Science, Technology, Engineering, and Mathematics)
	+ Occupation: Pr

🚀 Crew: crew
└── 📋 Task: 99e58ec7-290c-408f-81ac-e171db0c197d
       Status: Executing Task...
    └── 🤖 Agent: Content Planner
            Status: ✅ Completed

...

# Agent: Editor
## Task: Proofread the given blog post for grammatical errors and alignment with the brand's voice.


# Agent: Editor
## Final Answer: 
### Artificial Intelligence: Unlocking Human Potential or a Recipe for Disaster?

#### Introduction
Artificial Intelligence (AI) is transforming industries worldwide, from healthcare to finance, and beyond. With its ability to learn from data and improve over time, AI has the potential to revolutionize the way we live and work. However, as AI continues to advance, concerns about its impact on society are growing.

In this article, we'll explore the latest trends, key players, and noteworthy news in AI, helping you understand its potential and challenges. We'll examine what AI is, its various types, and how it's being applied in different industries. We'll also discuss the key players driving the AI landscape forward and highlight some of the most significant breakthroughs and developments.

#### What is Artificial Intelligence?

At its 

🚀 Crew: crew
├── 📋 Task: 99e58ec7-290c-408f-81ac-e171db0c197d
│      Assigned to: Content Planner
│      Status: ✅ Completed
│   └── 🤖 Agent: Content Planner
│           Status: ✅ Completed
├── 📋 Task: 45385638-e729-4245-a8ab-fa0683a53364
│      Assigned to: Content Writer
│      Status: ✅ Completed
│   └── 🤖 Agent: Content Writer
│           Status: ✅ Completed
└── 📋 Task: e05a388e-71d3-4a16-aca0-66ae52760be8
       Status: Executing Task...
    └── 🤖 Agent: Editor
            Status: ✅ Completed

🚀 Crew: crew
├── 📋 Task: 99e58ec7-290c-408f-81ac-e171db0c197d
│      Assigned to: Content Planner
│      Status: ✅ Completed
│   └── 🤖 Agent: Content Planner
│           Status: ✅ Completed
├── 📋 Task: 45385638-e729-4245-a8ab-fa0683a53364
│      Assigned to: Content Writer
│      Status: ✅ Completed
│   └── 🤖 Agent: Content Writer
│           Status: ✅ Completed
└── 📋 Task: e05a388e-71d3-4a16-aca0-66ae52760be8
       Assigned to: Editor
       Status: ✅ Completed
    └── 🤖 Agent: Editor
            Status: ✅ Completed

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: e05a388e-71d3-4a16-aca0-66ae52760be8                                                                     │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 22f09677-4656-48b9-9aa9-9f94c651ccf9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Showing the output in markdown format

In [39]:
Markdown(result.raw)

### Artificial Intelligence: Unlocking Human Potential or a Recipe for Disaster?

#### Introduction
Artificial Intelligence (AI) is transforming industries worldwide, from healthcare to finance, and beyond. With its ability to learn from data and improve over time, AI has the potential to revolutionize the way we live and work. However, as AI continues to advance, concerns about its impact on society are growing.

In this article, we'll explore the latest trends, key players, and noteworthy news in AI, helping you understand its potential and challenges. We'll examine what AI is, its various types, and how it's being applied in different industries. We'll also discuss the key players driving the AI landscape forward and highlight some of the most significant breakthroughs and developments.

#### What is Artificial Intelligence?

At its core, AI refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, problem-solving, and decision-making. The history of AI dates back to the 1950s when computer scientists began exploring ways to create machines that could think and learn like humans.

Today, there are several types of AI, including narrow or weak AI, which is designed to perform a specific task; general AI, which aims to replicate human intelligence in all domains; and superintelligence, which surpasses human intelligence in various areas. Understanding these different types of AI is crucial for grasping the potential applications and challenges associated with this technology.

#### Latest Trends in AI

One of the most significant trends in AI is explainability and transparency. As AI becomes more prevalent in industries such as healthcare and finance, there is a growing need to understand how AI-powered decisions are made. This requires developing methods to interpret and visualize AI's decision-making processes, enabling humans to trust and rely on these systems.

Another trend gaining momentum is edge AI, which involves deploying AI at the edge of the network, closer to where data is generated, rather than relying on centralized cloud computing. This approach enables faster processing times and reduced latency, making it ideal for applications such as smart cities and IoT integration.

Human-AI collaboration and augmentation are also becoming increasingly important as AI advances. By combining human creativity with AI's analytical capabilities, businesses can unlock new levels of innovation and productivity. For instance, AI can assist designers in creating more effective and efficient designs, while humans can provide the creative input and intuition needed to bring these ideas to life.

#### Key Players in the AI Landscape

The AI landscape is dominated by a few key players, including top research institutions and universities, major tech companies, and innovative startups. Some of the most notable research institutions include MIT-IBM Watson AI Lab [1] and Stanford Natural Language Processing Group [2]. These organizations are pushing the boundaries of AI research and development.

Major tech companies such as Google, Microsoft, and Amazon are also driving the AI landscape forward through their investments in AI research and infrastructure. Noteworthy startups and innovators, like DeepMind and Nuro, are exploring new applications for AI and developing cutting-edge technologies that have the potential to transform industries.

#### Noteworthy News and Developments

Recent breakthroughs in natural language processing (NLP) and computer vision have been particularly significant. For instance, researchers at Google's DeepMind lab developed an AI model called AlphaFold, which has shown remarkable success in predicting protein structures [3]. This breakthrough has the potential to revolutionize fields such as medicine and biotechnology.

Advances in AI for healthcare, finance, and education are also worth highlighting. In healthcare, AI is being used to develop personalized cancer treatments and improve medical diagnosis. In finance, AI-powered trading platforms are becoming increasingly popular, while in education, AI is helping teachers personalize learning experiences for students.

#### Conclusion

Artificial Intelligence has the potential to transform industries worldwide, but it also raises significant concerns about its impact on society. By understanding the latest trends, key players, and noteworthy news in AI, we can begin to unlock its full potential while addressing the challenges associated with this technology.

As we move forward, it's essential for businesses, governments, and individuals to work together to ensure that AI is developed and deployed responsibly. This requires ongoing investment in research and development, as well as education and training programs that equip people with the skills needed to thrive in an AI-driven world.

We encourage readers to explore AI further through online resources or attending industry events. Share your own experiences with AI and its applications in the comments section below.

### Resources

* MIT-IBM Watson AI Lab: [4]
* Stanford Natural Language Processing Group: [5]
* "Life 3.0: Being Human in the Age of Artificial Intelligence" by Max Tegmark: [6]
* "Deep Learning" by Ian Goodfellow, Yoshua Bengio, and Aaron Courville: [7]
* Coursera's Machine Learning course by Andrew Ng: [8]
* edX's AI for Everyone course by Andrew Moore: [9]

### Call to Action

Explore AI further through online resources or attending industry events. Share your own experiences with AI and its applications in the comments section below.

---

References:

[1] "A History of Artificial Intelligence" by David C. Dennett
[2] MIT-IBM Watson AI Lab: https://www.ibm.com/blogs/research/2018/04/mit-ibm-watson-ai-lab/
[3] "AlphaFold predicts the 3D structure of proteins with high accuracy" by DeepMind
[4] MIT-IBM Watson AI Lab: https://www.ibm.com/blogs/research/2018/04/mit-ibm-watson-ai-lab/
[5] Stanford Natural Language Processing Group: https://nlp.stanford.edu/
[6] "Life 3.0: Being Human in the Age of Artificial Intelligence" by Max Tegmark
[7] "Deep Learning" by Ian Goodfellow, Yoshua Bengio, and Aaron Courville
[8] Coursera's Machine Learning course by Andrew Ng: https://www.coursera.org/specializations/machine-learning
[9] edX's AI for Everyone course by Andrew Moore: https://www.edx.org/course/ai-for-everyone